# 01 — Exploratory Data Analysis: AG News

Notebook này khám phá dataset AG News trước khi đưa vào pipeline.
Mục tiêu:
- Phân phối nhãn
- Phân phối độ dài văn bản
- Top từ theo từng lớp
- Kiểm tra class imbalance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from collections import Counter
from datasets import load_dataset

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

LABEL_NAMES = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

In [ ]:
# Load dataset
hf = load_dataset('ag_news')
train_df = hf['train'].to_pandas()
test_df  = hf['test'].to_pandas()

train_df['split'] = 'train'
test_df['split']  = 'test'
df = pd.concat([train_df, test_df], ignore_index=True)

df['label_name']  = df['label'].map(LABEL_NAMES)
df['text_length'] = df['text'].str.len()
df['word_count']  = df['text'].str.split().str.len()

print(f'Train: {len(train_df):,} | Test: {len(test_df):,} | Total: {len(df):,}')
df.head(3)

## 1. Phân phối nhãn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Train
counts_train = train_df['label'].value_counts().sort_index()
axes[0].bar(
    [LABEL_NAMES[i] for i in counts_train.index],
    counts_train.values,
    color=COLORS
)
axes[0].set_title('Train set label distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts_train.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9)

# Test
counts_test = test_df['label'].value_counts().sort_index()
axes[1].bar(
    [LABEL_NAMES[i] for i in counts_test.index],
    counts_test.values,
    color=COLORS
)
axes[1].set_title('Test set label distribution')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('results/figures/label_distribution.png', bbox_inches='tight')
plt.show()

# Kiểm tra class balance
print('\nClass balance (train):')
print((counts_train / counts_train.sum() * 100).round(2).to_string())

## 2. Phân phối độ dài văn bản

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram word count
for label_id, name in LABEL_NAMES.items():
    subset = train_df[train_df['label'] == label_id]['word_count']
    subset = subset[subset < subset.quantile(0.99)]  # loại outlier
    axes[0].hist(subset, bins=40, alpha=0.5, label=name, color=COLORS[label_id])

axes[0].set_xlabel('Word count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Word count distribution by class')
axes[0].legend()

# Box plot
data = [train_df[train_df['label'] == i]['word_count'].values for i in range(4)]
bp = axes[1].boxplot(data, labels=list(LABEL_NAMES.values()), patch_artist=True)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

axes[1].set_ylabel('Word count')
axes[1].set_title('Word count boxplot by class')

plt.tight_layout()
plt.savefig('results/figures/text_length_distribution.png', bbox_inches='tight')
plt.show()

# Thống kê mô tả
print(train_df.groupby('label_name')['word_count'].describe().round(1))

## 3. Token length vs BERT max_length=128

In [ ]:
# Kiểm tra bao nhiêu % mẫu bị truncate khi dùng max_length=128
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Lấy mẫu 2000 rows để tính nhanh
sample = train_df.sample(2000, random_state=42)
token_counts = sample['text'].apply(
    lambda t: len(tokenizer.encode(t, truncation=False))
)

pct_truncated = (token_counts > 128).mean() * 100

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(token_counts, bins=50, color='#4C72B0', alpha=0.8, edgecolor='white')
ax.axvline(128, color='red', linestyle='--', linewidth=1.5, label='max_length=128')
ax.set_xlabel('Token count (BERT tokenizer)')
ax.set_ylabel('Frequency')
ax.set_title(f'Token length distribution — {pct_truncated:.1f}% samples truncated at 128')
ax.legend()
plt.tight_layout()
plt.savefig('results/figures/token_length.png', bbox_inches='tight')
plt.show()

print(f'Median token count : {token_counts.median():.0f}')
print(f'90th percentile    : {token_counts.quantile(0.9):.0f}')
print(f'% truncated at 128 : {pct_truncated:.1f}%')

## 4. Top từ theo từng lớp (TF-IDF style)

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

STOP_WORDS = 'english'
TOP_N = 12

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for label_id, ax in enumerate(axes):
    label_name = LABEL_NAMES[label_id]
    texts = train_df[train_df['label'] == label_id]['text'].tolist()

    tfidf = TfidfVectorizer(
        max_features=5000,
        stop_words=STOP_WORDS,
        ngram_range=(1, 1),
    )
    tfidf.fit(texts)

    scores  = tfidf.idf_
    vocab   = tfidf.get_feature_names_out()

    # IDF thấp = xuất hiện nhiều trong corpus → đặc trưng cho lớp này
    top_idx   = scores.argsort()[:TOP_N]
    top_words = [vocab[i] for i in top_idx]
    top_scores = [scores[i] for i in top_idx]

    ax.barh(top_words[::-1], top_scores[::-1], color=COLORS[label_id], alpha=0.8)
    ax.set_title(label_name)
    ax.set_xlabel('IDF score (lower = more frequent)')

plt.suptitle('Top characteristic words per class (low IDF = high frequency)', y=1.02)
plt.tight_layout()
plt.savefig('results/figures/top_words_per_class.png', bbox_inches='tight')
plt.show()

## 5. Summary cho báo cáo

In [ ]:
import json

eda_summary = {
    'dataset':           'AG News',
    'n_train':           len(train_df),
    'n_test':            len(test_df),
    'n_classes':         4,
    'class_names':       list(LABEL_NAMES.values()),
    'is_balanced':       True,
    'train_samples_per_class': (counts_train / len(train_df)).round(4).to_dict(),
    'median_word_count': float(train_df['word_count'].median()),
    'pct_truncated_128': round(pct_truncated, 2),
}

import os
os.makedirs('results/metrics', exist_ok=True)
with open('results/metrics/eda_summary.json', 'w') as f:
    json.dump(eda_summary, f, indent=2)

print('EDA summary:')
for k, v in eda_summary.items():
    print(f'  {k:<30} {v}')